In [1]:
import sqlite3 as sql
import pandas as pd

In [2]:
# Conexão com banco de dados relacional em memória (RDBMS)
conn = sql.connect(":memory:")

In [3]:
# 1. Copiar tabelas do projeto para o Banco de Dados SQL (Simulação Sqoop Export)
df_vendas = pd.read_csv('../vendas.csv')
df_clientes = pd.read_csv('../clientes.csv')

df_vendas[['id_venda', 'id_cliente', 'produto', 'categoria', 'quantidade', 'preco_unitario', 'canal']].to_sql("vendas_sql", conn, index=False)
df_clientes[['id_cliente', 'idade', 'cidade', 'canal_preferencial']].to_sql("clientes_sql", conn, index=False)

612

In [4]:
# 2. Leitura filtrada do SQL via Sqoop Import com consulta SELECT
df_sqoop_online = pd.read_sql("SELECT * FROM vendas_sql WHERE canal = 'Online'", conn)
print(f"[Sqoop] Vendas no canal Online importadas: {len(df_sqoop_online)} registros")

[Sqoop] Vendas no canal Online importadas: 4628 registros


In [5]:
# 3. Consulta complexa agrupando vendas por cidade do cliente
df_sqoop_agregado = pd.read_sql("""
    SELECT c.cidade, SUM(v.quantidade) AS total_unidades
    FROM vendas_sql v
    JOIN clientes_sql c ON v.id_cliente = c.id_cliente
    GROUP BY c.cidade
    ORDER BY total_unidades DESC
""", conn)

In [6]:
# 4. Exportação do resultado em lote para arquivo CSV (Data Lake / HDFS)
df_sqoop_agregado.to_csv('vendas_por_cidade_sqoop.csv', index=False)
print("[Sqoop] Exportação concluída com sucesso para 'vendas_por_cidade_sqoop.csv'.")

[Sqoop] Exportação concluída com sucesso para 'vendas_por_cidade_sqoop.csv'.
